# Day-Ahead Electricity Demand Forecasting (+24h Horizon)

This notebook extends a baseline short-term forecasting pipeline to a **day-ahead electricity demand forecasting problem**.

The objective is to evaluate how predictive performance changes when:
- the forecasting horizon is shifted from +1h to +24h
- exogenous weather variables (temperature) are introduced
- statistical and machine learning models are compared under realistic constraints

## Workflow

1. Load and preprocess electricity + weather data  
2. Align and validate time-series consistency  
3. Construct lag features with +24h target shift  
4. Add weather-based exogenous variables  
5. Train baseline, statistical, and ML models  
6. Evaluate them with a chronological train/test split  
7. Validate robustness with time-series cross-validation 

## Pipeline overview

Raw data → Time Alignment → Feature Engineering → Target Shift (+24h) → Train/Test Split → Model Training → Evaluation

## Models compared

### Baselines
- Naive forecast (t-24)

### Statistical models
- Exponential Smoothing (Holt-Winters)

### Machine learning models
- Ridge Regression
- Random Forest Regressor
- Gradient Boosting Regressor

The goal is to move from a simplified forecasting setup to a more realistic **day-ahead electricity demand forecasting pipeline**, where predictive performance is evaluated under a +24h horizon and enriched with exogenous weather information.

Beyond accuracy, the focus is on model robustness, interpretability, and alignment with real-world energy forecasting constraints.

## Configuration

In [1]:
# =========================
# Project configuration
# =========================

DATA_PATH = "../data/PJME_hourly.csv"
TARGET_COL = "PJME_MW"

HORIZON = 24  # day-ahead forecasting

TEST_SIZE = 0.2
RANDOM_STATE = 42

# plotting window (first month)
PLOT_HOURS = 24 * 30

In [2]:
# ============================================================================
# Weather configuration: PJM East approximate center --> Philadelphia area
# ============================================================================

LAT = 39.95
LON = -75.16

## Imports

In [3]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [4]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit

from src.preprocessing import load_data, sort_by_time, run_data_checks
from src.weather import fetch_open_meteo_weather
from src.features import create_features
from src.train import (
    make_Xy,
    make_train_test_split,
    build_ridge_pipeline,
    build_random_forest_model,
    build_gradient_boosting_model,
    fit_predict,
)
from src.evaluate import evaluate_regression

## Data loading and preprocessing

I load the electricity demand dataset, standardize the timestamp format, convert it to datetime, and sort the series chronologically to ensure proper time-series structure for forecasting.

In [5]:
df = load_data("../data/PJME_hourly.csv")
df = sort_by_time(df)

print("Raw shape:", df.shape)
df.head()

Raw shape: (145366, 1)


,PJME_MW
timestamp,
2002-01-01 01:00:00,30393.0
2002-01-01 02:00:00,29265.0
2002-01-01 03:00:00,28357.0
2002-01-01 04:00:00,27899.0
2002-01-01 05:00:00,28057.0


## Data quality checks

Before feature engineering, I run a set of basic validation checks to ensure the time series is reliable and suitable for forecasting. These checks include:

- ordering and continuity of timestamps  
- detection of duplicates and missing values  
- verification of expected hourly frequency  
- assessment of missing time steps in the full time range   

These steps are critical because forecasting models rely on consistent temporal spacing for lag and rolling features.

In [6]:
checks, time_deltas = run_data_checks(df)

print("Data checks")
for name, value in checks.items():
    print(f"- {name}: {value}")

print("\nMost common time deltas:")
print(time_deltas)

Data checks
- sorted_index: True
- duplicate_timestamps: 4
- missing_target_values: 0
- negative_target_values: 0
- missing_timestamps: 26

Most common time deltas:
timestamp
0 days 01:00:00    145331
0 days 02:00:00        30
0 days 00:00:00         4
Name: count, dtype: int64


In [7]:
# Remove duplicated timestamps, if any, keeping the first occurrence
df = df[~df.index.duplicated(keep="first")].copy()

print("Shape after duplicate handling:", df.shape)

Shape after duplicate handling: (145362, 1)


## Target definition (+24h forecasting horizon)

To transform the problem into a supervised learning task, I shift the target variable by 24 hours ahead. This means that for each timestamp t, the model learns to predict the electricity demand at t + 24 hours.

This setup ensures a true **day-ahead forecasting scenario**, commonly used in energy markets and grid planning, and avoids data leakage, in the sense that all features must only contain information available at or before time t.

In [8]:
df = df.sort_index()

df["target"] = df[TARGET_COL].shift(-24)

print("Original shape:", df.shape)
print("Missing target values (expected at the end):", df["target"].isna().sum())

# Remove last 24 rows (no target available)
df = df.dropna(subset=["target"])

print("Shape after target alignment:", df.shape)

Original shape: (145362, 2)
Missing target values (expected at the end): 24
Shape after target alignment: (145338, 2)


The last 24 rows are removed because they do not have a valid target value after shifting.
This is expected and does not represent missing data issues.

## Weather data integration (Open-Meteo API)

To improve the realism and predictive power of the model, external weather data is incorporated.

Electricity demand is strongly influenced by temperature, especially due to heating and cooling needs. Therefore, temperature is used as an exogenous variable.

I use the Open-Meteo API to retrieve historical hourly temperature data aligned with the PJM electricity demand timestamps.

### Why weather data matters

- Captures seasonal consumption effects
- Improves model performance during extreme temperatures
- Adds exogenous explanatory power beyond autocorrelation

## Weather-energy data alignment

Weather data is merged with electricity demand using timestamp alignment. This ensures:
- no leakage
- correct temporal matching
- consistent hourly frequency

In [9]:
start_date = df.index.min().date()
end_date = df.index.max().date()

weather_df = fetch_open_meteo_weather(
    latitude=LAT,
    longitude=LON,
    start_date=start_date,
    end_date=end_date,
    timezone="UTC"
)

df = df.reset_index().rename(columns={"index": "timestamp"})
df = df.merge(weather_df, on="timestamp", how="left")
df = df.set_index("timestamp")

print(df[["PJME_MW", "temperature"]].head())

                     PJME_MW  temperature
timestamp                                
2002-01-01 01:00:00  30393.0         -3.5
2002-01-01 02:00:00  29265.0         -3.7
2002-01-01 03:00:00  28357.0         -3.8
2002-01-01 04:00:00  27899.0         -4.1
2002-01-01 05:00:00  28057.0         -4.4


In [10]:
print("Missing temperature values:", df["temperature"].isna().sum())

print(df[["PJME_MW", "temperature"]].describe())

Missing temperature values: 0
             PJME_MW    temperature
count  145338.000000  145338.000000
mean    32079.281723      12.855045
std      6463.212290      10.288486
min     14544.000000     -21.900000
25%     27573.000000       4.600000
50%     31421.000000      13.400000
75%     35647.000000      21.400000
max     62009.000000      39.900000


## Feature engineering

To improve forecasting performance, a comprehensive set of engineered features is created to capture both internal demand dynamics and external drivers.

The feature set includes:

- **Temporal features**: hour of day, day of week, month, and weekend indicators
- **Cyclical encoding**: sine/cosine transformations to properly represent periodic time structures
- **Autoregressive components**: lagged electricity demand values to capture short- and long-term persistence
- **Rolling statistics**: moving averages and variability measures to model local trends and volatility
- **Weather-based features**: temperature and derived indicators to capture heating and cooling effects on demand

All features are strictly computed using past information only, ensuring no data leakage and preserving the causal structure required in time-series forecasting.

Feature engineering is applied after target alignment (+24h shift) to ensure a realistic day-ahead forecasting setup.

In [11]:
df_feat = create_features(df)

print("Feature matrix shape:", df_feat.shape)
df_feat.head()

Feature matrix shape: (145170, 21)


,PJME_MW,target,temperature,hour,dayofweek,month,is_weekend,hour_sin,hour_cos,dow_sin,...,lag_1,lag_24,lag_168,rolling_mean_24,rolling_std_24,temp_lag_1,temp_lag_24,temp_roll_mean_24,heating_degree,cooling_degree
timestamp,,,,,,,,,,,,,,,,,,,,,
2002-01-08 01:00:00,29445.0,29082.0,-1.2,1,1,1,0,0.258819,0.965926,0.781831,...,31187.0,26862.0,30393.0,33452.583333,4559.767709,-0.5,1.0,0.958333,19.2,0.0
2002-01-08 02:00:00,28670.0,28154.0,-2.1,2,1,1,0,0.500000,0.866025,0.781831,...,29445.0,25976.0,29265.0,33560.208333,4425.965952,-1.2,1.0,0.829167,20.1,0.0
2002-01-08 03:00:00,28375.0,27829.0,-2.8,3,1,1,0,0.707107,0.707107,0.781831,...,28670.0,25641.0,28357.0,33672.458333,4256.159403,-2.1,0.9,0.675000,20.8,0.0
2002-01-08 04:00:00,28542.0,27759.0,-3.7,4,1,1,0,0.866025,0.500000,0.781831,...,28375.0,25666.0,27899.0,33786.375000,4064.104959,-2.8,1.0,0.479167,21.7,0.0
2002-01-08 05:00:00,29261.0,28308.0,-4.7,5,1,1,0,0.965926,0.258819,0.781831,...,28542.0,26328.0,28057.0,33906.208333,3851.076461,-3.7,1.2,0.233333,22.7,0.0


After feature construction, the resulting dataset is fully aligned for supervised learning. Each row contains only information available up to time *t*, while the target variable represents electricity demand at time *t + 24h*.

This ensures a realistic day-ahead forecasting framework and prevents information leakage from future observations.